# Lecture 8: Common Data Formats

As a computational linguist or data scientist, you'll constantly work with data stored in files. Different formats suit different kinds of data:

- **Tabular data** (rows and columns) → CSV
- **Hierarchical/nested data** (trees, records with varying fields) → JSON, YAML, XML

This notebook covers the four most common formats you'll encounter: **CSV**, **JSON**, **YAML**, and **XML**.

**Prerequisites:**
- Python and the `ling250` environment
- Demo files in `../demos/data_formats/`

## Setup

This lecture introduces `pandas`, a new library for working with tabular data. We've added it to the course `environment.yaml`, so you need to **update your environment** before running this notebook.

In your terminal (with the `ling250` environment activated), run:

```bash
conda env update -f environment.yaml --prune
```

This reads the environment file and installs any packages that were added since you last created/updated the environment. The `--prune` flag removes packages that are no longer listed, keeping your environment in sync with the file.

This is the same workflow you'd use any time a project's environment file changes — a common situation when collaborating with others via git.

Once your environment is updated, import the libraries we'll use:

In [ ]:
import csv
import json
import yaml
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../demos/data_formats")

---

## Part 1: CSV (Comma-Separated Values)

CSV is the most common format for **tabular data** — think spreadsheets, experimental results, corpus metadata, or formant measurements.

Each line is a row, and columns are separated by commas. The first line is usually a header.

### What CSV looks like (raw)

Let's look at the raw text of our demo file:

In [ ]:
csv_path = DATA_DIR / "example.csv"

raw_text = csv_path.read_text()
print(raw_text)

### Reading CSV with pandas

`pandas` is the standard library for tabular data in Python. Its `read_csv()` function handles parsing, type inference, and much more.

In [ ]:
df = pd.read_csv(csv_path)
df

Notice that pandas automatically:
- Used the first row as column names
- Inferred types (integers for `age` and `year`, floats for `gpa`)
- Created an index (the numbers on the left)

In [ ]:
# Access a single column
df["name"]

In [ ]:
# Filter rows
df[df["gpa"] > 3.9]

### Writing CSV

You can write a DataFrame back to CSV. The `index=False` parameter prevents pandas from writing its row numbers as a column.

In [ ]:
output_path = DATA_DIR / "output_example.csv"
df.to_csv(output_path, index=False)

# Verify by reading it back
print(output_path.read_text())

### CSV gotchas

**Commas inside fields:** If a value itself contains a comma, it must be quoted.

In [ ]:
# pandas handles quoted fields automatically
tricky_csv = 'name,description\nCecil,"Radio host, community voice"\nCarlos,Scientist\n'
print("Raw CSV:")
print(tricky_csv)

import io
print("Parsed:")
pd.read_csv(io.StringIO(tricky_csv))

**TSV (Tab-Separated Values):** Some files use tabs instead of commas. Just pass `sep="\t"` to `read_csv()`.

In [ ]:
tsv_text = "name\tage\nCecil\t35\nDana\t23\n"
pd.read_csv(io.StringIO(tsv_text), sep="\t")

**Encoding:** Older files sometimes use `latin-1` instead of `utf-8`. If you get garbled text, try `pd.read_csv(path, encoding="latin-1")`.

---

## Part 2: JSON (JavaScript Object Notation)

JSON is the standard format for **hierarchical/nested data**. You'll encounter it in NLP tool outputs, API responses, and web-scraped data.

JSON maps directly to Python data structures:

| JSON | Python |
|------|--------|
| `{}` object | `dict` |
| `[]` array | `list` |
| `"string"` | `str` |
| `123` / `1.5` | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

### What JSON looks like (raw)

In [ ]:
json_path = DATA_DIR / "example.json"

print(json_path.read_text())

### Reading JSON

Python's built-in `json` module has two reading functions:
- `json.load(file)` — reads from an open file
- `json.loads(string)` — reads from a string

(Think: `load` from a **s**tream vs. `loads` from a **s**tring.)

In [ ]:
# load from file
with open(json_path) as f:
    data = json.load(f)

print(type(data))
print(data.keys())

In [ ]:
# loads from string
json_string = '{"name": "Cecil", "age": 35}'
parsed = json.loads(json_string)
print(parsed)
print(type(parsed))

### Navigating nested structures

Once loaded, JSON is just Python dicts and lists. Navigate with indexing:

In [ ]:
characters = data["nightvale_characters"]

# First character's name
print(characters[0]["name"])

# First character's friends
print(characters[0]["friends"])

# All character names
for character in characters:
    print(f"{character['name']}: {character['major']}")

### Writing JSON

The writing functions mirror the reading ones:
- `json.dump(obj, file)` — writes to a file
- `json.dumps(obj)` — returns a string

The `indent` parameter makes output human-readable:

In [ ]:
# Without indent (compact)
print(json.dumps(parsed))

# With indent (pretty)
print(json.dumps(parsed, indent=2))

In [ ]:
# Write to a file
output_path = DATA_DIR / "output_example.json"
with open(output_path, "w") as f:
    json.dump(parsed, f, indent=2)

print(output_path.read_text())

### JSON Lines

Sometimes you'll see files where each line is a separate JSON object. This is called **JSON Lines** (`.jsonl`). It's common in NLP datasets because you can process one record at a time without loading the whole file.

In [ ]:
jsonl_example = '{"name": "Cecil", "age": 35}\n{"name": "Dana", "age": 23}\n'
print("Raw JSONL:")
print(jsonl_example)

# Read by parsing each line
records = [json.loads(line) for line in jsonl_example.strip().split("\n")]
print("Parsed:")
print(records)

---

## Part 3: YAML (YAML Ain't Markup Language)

YAML is designed to be **human-readable and human-writable**. It's the standard for configuration files — including Conda environment files.

YAML uses indentation (like Python!) instead of braces and brackets.

### What YAML looks like

In [ ]:
yaml_path = DATA_DIR / "example.yaml"

print(yaml_path.read_text())

### YAML vs JSON

YAML and JSON can represent the same data. Compare:

```json
{"name": "Cecil", "friends": ["Carlos", "Dana"]}
```

```yaml
name: Cecil
friends:
  - Carlos
  - Dana
```

YAML is easier to read and edit by hand. JSON is easier to parse by machines.

### Reading YAML

The `pyyaml` library (already in your `ling250` environment) provides:
- `yaml.safe_load(file_or_string)` — parses YAML into Python objects

**Always use `safe_load`**, not `load`. The unsafe version can execute arbitrary code!

In [ ]:
with open(yaml_path) as f:
    yaml_data = yaml.safe_load(f)

print(type(yaml_data))
print(yaml_data)

In [ ]:
# Access like any Python dict
print(yaml_data["Cecil"])
print(yaml_data["Cecil"]["major"])

### Writing YAML

In [ ]:
# safe_dump converts Python objects to a YAML string
print(yaml.safe_dump(yaml_data["Cecil"]))

In [ ]:
# Write to a file
output_path = DATA_DIR / "output_example.yaml"
with open(output_path, "w") as f:
    yaml.safe_dump(yaml_data, f)

print(output_path.read_text())

### When you'll see YAML

YAML is everywhere in configuration:
- **Conda environments** — your own `environment.yaml` is YAML! Let's look at it:

In [ ]:
env_path = Path("../environment.yaml")

with open(env_path) as f:
    env_data = yaml.safe_load(f)

print(f"Environment name: {env_data['name']}")
print(f"Number of dependencies: {len(env_data['dependencies'])}")
print(f"Dependencies: {env_data['dependencies']}")

---

## Part 4: XML (eXtensible Markup Language)

XML uses **tags** (like HTML) to structure data. You'll encounter it in:
- **TEI-encoded texts** (historical documents, literary editions)
- **Linguistic annotation formats** (we'll see more of this in upcoming lectures)
- Older web APIs and data archives

XML has three key concepts:
- **Elements:** defined by opening and closing tags (`<name>Cecil</name>`)
- **Attributes:** metadata on elements (`<student id="1">`)
- **Nesting:** elements contain other elements

### What XML looks like

In [ ]:
xml_path = DATA_DIR / "example.xml"

print(xml_path.read_text())

Notice the structure:
- `<students>` is the **root element** that contains everything
- Each `<student>` has **attributes** (`id`, `status`) and **child elements** (`<name>`, `<age>`, etc.)
- `<friends/>` is a **self-closing tag** (empty element — Hiram has no friends)

### Parsing XML

Python's standard library includes `xml.etree.ElementTree` for parsing XML:

In [ ]:
tree = ET.parse(xml_path)
root = tree.getroot()

print(f"Root tag: {root.tag}")
print(f"Number of children: {len(root)}")

### Finding elements

Use `.find()` for the first match or `.findall()` for all matches:

In [ ]:
# Get all student elements
students = root.findall("student")

for student in students:
    name = student.find("name").text
    major = student.find("major").text
    print(f"{name}: {major}")

### Extracting attributes

In [ ]:
for student in students:
    name = student.find("name").text
    student_id = student.get("id")
    status = student.get("status")
    print(f"ID {student_id}: {name} ({status})")

### Navigating nested elements

In [ ]:
for student in students:
    name = student.find("name").text
    friends_elem = student.find("friends")
    friend_names = [f.text for f in friends_elem.findall("friend")]
    print(f"{name}'s friends: {friend_names}")

### Why XML matters for linguistics

XML is the foundation for many linguistic data formats:
- **TEI (Text Encoding Initiative):** standard for encoding historical/literary texts
- **Annotation formats:** many tools export linguistic annotations as XML

We'll work with some of these in upcoming lectures.

---

## Part 5: Comparison

| Format | Best for | Python module | Human-readable? | Supports nesting? |
|--------|----------|---------------|-----------------|--------------------|
| CSV | Tabular data (rows & columns) | `pandas` / `csv` | Yes (simple data) | No |
| JSON | Nested/structured data | `json` (built-in) | Somewhat | Yes |
| YAML | Configuration files | `pyyaml` | Very | Yes |
| XML | Annotated/marked-up text | `xml.etree` (built-in) | Somewhat | Yes |

### Decision guide

- **Spreadsheet-like data?** → CSV
- **Records with varying fields?** → JSON
- **Config meant for humans to edit?** → YAML
- **Text with markup/annotations?** → XML

---

## Part 6: Converting Between Formats

A common real-world task: read data in one format, write it in another. Since all formats become Python objects once loaded, you just load → transform → save.

### CSV → JSON

In [ ]:
# Read CSV into a DataFrame, then convert to a list of dicts
df = pd.read_csv(DATA_DIR / "example.csv")
records = df.to_dict(orient="records")
print(json.dumps(records, indent=2))

### JSON → CSV

In [ ]:
# The flat fields (not "friends") map nicely to columns
with open(DATA_DIR / "example.json") as f:
    json_data = json.load(f)

characters = json_data["nightvale_characters"]

# Create DataFrame from the flat fields
flat_records = [
    {"name": c["name"], "age": c["age"], "year": c["year"],
     "major": c["major"], "gpa": c["gpa"]}
    for c in characters
]
df_from_json = pd.DataFrame(flat_records)
print(df_from_json.to_csv(index=False))

Notice we had to drop `friends` — **nested data doesn't fit neatly into a table.** This is the key tradeoff between tabular and hierarchical formats.

### XML → JSON

In [ ]:
tree = ET.parse(DATA_DIR / "example.xml")
root = tree.getroot()

characters_from_xml = []
for student in root.findall("student"):
    record = {
        "name": student.find("name").text,
        "age": int(student.find("age").text),
        "year": int(student.find("year").text),
        "major": student.find("major").text,
        "gpa": float(student.find("gpa").text),
        "friends": [
            f.text
            for f in student.find("friends").findall("friend")
        ],
    }
    characters_from_xml.append(record)

print(json.dumps(characters_from_xml, indent=2))

---

## Challenges

Try these on your own before looking at the solutions!

### Challenge 1: Highest GPA from CSV

Read `example.csv` and find the student with the highest GPA. Print their name and GPA.

**Hint:** Look at `DataFrame.sort_values()` or `DataFrame.loc` with `idxmax()`.

In [ ]:
# Your code here


### Challenge 2: JSON friend network

Read `example.json` and build a dictionary mapping each character's name to their number of friends. Handle the case where `friends` is `null`.

**Hint:** Check for `None` before calling `len()`.

In [ ]:
# Your code here


### Challenge 3: XML to CSV

Read `example.xml` and write a CSV file containing only the active students (where `status="active"`).

**Hint:** Use `student.get("status")` to read the attribute, and `pd.DataFrame` to build the table.

In [ ]:
# Your code here


### Challenge 4: Round-trip conversion

Read `example.yaml`, convert it to JSON format (as a string), and print it with nice indentation.

**Hint:** YAML loads into Python dicts/lists, and `json.dumps()` can serialize those.

In [ ]:
# Your code here


---

## Quick Reference

### Reading data

| Format | Code |
|--------|------|
| CSV | `df = pd.read_csv(path)` |
| JSON | `data = json.load(open(path))` |
| YAML | `data = yaml.safe_load(open(path))` |
| XML | `tree = ET.parse(path); root = tree.getroot()` |

### Writing data

| Format | Code |
|--------|------|
| CSV | `df.to_csv(path, index=False)` |
| JSON | `json.dump(data, open(path, "w"), indent=2)` |
| YAML | `yaml.safe_dump(data, open(path, "w"))` |

### Key differences

| | CSV | JSON | YAML | XML |
|---|---|---|---|---|
| **Data shape** | Tabular | Hierarchical | Hierarchical | Hierarchical |
| **Standard library?** | `csv` | `json` ✓ | No (`pyyaml`) | `xml.etree` ✓ |
| **Common in** | Data science | Web APIs, NLP | Config files | Text encoding |

---

## Challenge Solutions

Scroll down only after you've attempted the challenges yourself!

<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Highest GPA from CSV

In [ ]:
df = pd.read_csv(DATA_DIR / "example.csv")
best_idx = df["gpa"].idxmax()
best = df.loc[best_idx]
print(f"{best['name']} has the highest GPA: {best['gpa']}")

### Solution 2: JSON friend network

In [ ]:
with open(DATA_DIR / "example.json") as f:
    data = json.load(f)

friend_counts = {}
for character in data["nightvale_characters"]:
    friends = character["friends"]
    friend_counts[character["name"]] = len(friends) if friends is not None else 0

print(friend_counts)

### Solution 3: XML to CSV (active students only)

In [ ]:
tree = ET.parse(DATA_DIR / "example.xml")
root = tree.getroot()

active_records = []
for student in root.findall("student"):
    if student.get("status") == "active":
        active_records.append({
            "name": student.find("name").text,
            "age": int(student.find("age").text),
            "year": int(student.find("year").text),
            "major": student.find("major").text,
            "gpa": float(student.find("gpa").text),
        })

active_df = pd.DataFrame(active_records)
print(active_df)
print()
print(active_df.to_csv(index=False))

### Solution 4: Round-trip conversion (YAML → JSON)

In [ ]:
with open(DATA_DIR / "example.yaml") as f:
    yaml_data = yaml.safe_load(f)

print(json.dumps(yaml_data, indent=2))